# Custom refit strategy of a grid search with cross-validation (scikit-learn vs Moxie)

**Reference example:** `../scikit-learn/examples/model_selection/plot_grid_search_digits.py` at scikit-learn
`4358542a1bd39af2e7e1cbace8b9fed067473c5b`
([gallery page](https://scikit-learn.org/dev/auto_examples/model_selection/plot_grid_search_digits.html)).

**Scenario:** binary "is it an 8?" on the bundled digits (no download), `train_test_split(test_size=0.5,
random_state=0)`, `GridSearchCV(SVC(), tuned_parameters, scoring=["precision", "recall"],
refit=refit_strategy)` over two sub-grids (`rbf` × `gamma ∈ {1e-3, 1e-4}` × `C ∈ {1, 10, 100, 1000}`
and `linear` × `C`), then a classification report of the refitted model on the held-out half.
This exercises the search's list-of-grids `ParameterGrid`, multi-metric scoring, ranks and a callable
refit policy, over an estimator of another family (svm).

**Adaptations:**

1. Labels: sklearn uses `digits.target == 8` (bool); Moxie uses `SVC[Int]` with 0/1 (typed label
   domain, documented). Positive label 1 ↔ `True`.
2. `scoring=["precision", "recall"]`: Moxie has no scorer strings or `moxie.metrics` (shared_surface #31);
   one typed `SearchScorer` computes binary precision (`zero_division` → 0.0, like sklearn's default
   "warn" value) or recall, and is passed twice with `metric_names=["precision", "recall"]`.
3. Mixed-type grid values (`kernel` strings next to floats): Moxie's grid has one `Value` type, so the
   adapter encodes `kernel` as 0.0 = `"rbf"`, 1.0 = `"linear"` (documented typed-adapter difference).
4. `refit=refit_strategy`: the same filter-and-pick logic as a `SearchRefitPolicy` struct. Its last step
   picks the candidate with the smallest `mean_score_time`, a wall-clock measurement, so the final
   choice is **not deterministic on either side**. We compare the deterministic part exactly (the
   candidate set that survives the precision and recall filters) and then compare Moxie's refitted model
   against sklearn's `SVC` with the parameters Moxie chose.
5. pandas `DataFrame` printing is replaced by NumPy arrays.

**Determinism:** everything except the timing-based final pick is deterministic: `train_test_split`
with a fixed seed (the indices are computed by sklearn and shared), unshuffled `StratifiedKFold(5)`,
LIBSVM without probability. Scores and predictions are compared exactly.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("cross_family/plot_grid_search_digits")

## 1. scikit-learn (official example code)

In [2]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split

digits = datasets.load_digits()
n_samples = len(digits.images)
X = digits.images.reshape((n_samples, -1))
y = digits.target == 8
print(f"The number of images is {X.shape[0]} and each image contains {X.shape[1]} pixels")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)

scores = ["precision", "recall"]


def print_dataframe(filtered_cv_results):
    """Pretty print for filtered dataframe"""
    for mean_precision, std_precision, mean_recall, std_recall, params in zip(
        filtered_cv_results["mean_test_precision"],
        filtered_cv_results["std_test_precision"],
        filtered_cv_results["mean_test_recall"],
        filtered_cv_results["std_test_recall"],
        filtered_cv_results["params"],
    ):
        print(
            f"precision: {mean_precision:0.3f} (±{std_precision:0.03f}),"
            f" recall: {mean_recall:0.3f} (±{std_recall:0.03f}),"
            f" for {params}"
        )
    print()


survivors = {}


def refit_strategy(cv_results):
    """Define the strategy to select the best estimator (official code, printing trimmed)."""
    precision_threshold = 0.98
    cv_results_ = pd.DataFrame(cv_results)
    print("All grid-search results:")
    print_dataframe(cv_results_)
    high_precision_cv_results = cv_results_[cv_results_["mean_test_precision"] > precision_threshold]
    high_precision_cv_results = high_precision_cv_results[
        ["mean_score_time", "mean_test_recall", "std_test_recall", "mean_test_precision",
         "std_test_precision", "rank_test_recall", "rank_test_precision", "params"]
    ]
    best_recall_std = high_precision_cv_results["mean_test_recall"].std()
    best_recall = high_precision_cv_results["mean_test_recall"].max()
    best_recall_threshold = best_recall - best_recall_std
    high_recall_cv_results = high_precision_cv_results[
        high_precision_cv_results["mean_test_recall"] > best_recall_threshold
    ]
    print("Models within one standard deviation of the highest recall, among precision > 0.98:")
    print_dataframe(high_recall_cv_results)
    survivors["index"] = np.array(sorted(high_recall_cv_results.index))
    return high_recall_cv_results["mean_score_time"].idxmin()


from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

tuned_parameters = [
    {"kernel": ["rbf"], "gamma": [1e-3, 1e-4], "C": [1, 10, 100, 1000]},
    {"kernel": ["linear"], "C": [1, 10, 100, 1000]},
]
grid_search = GridSearchCV(SVC(), tuned_parameters, scoring=scores, refit=refit_strategy)
grid_search.fit(X_train, y_train)
print("best_params_:", grid_search.best_params_)

from sklearn.metrics import classification_report
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))

The number of images is 1797 and each image contains 64 pixels


All grid-search results:
precision: 1.000 (±0.000), recall: 0.854 (±0.063), for {'C': 1, 'gamma': 0.001, 'kernel': 'rbf'}
precision: 1.000 (±0.000), recall: 0.257 (±0.061), for {'C': 1, 'gamma': 0.0001, 'kernel': 'rbf'}
precision: 1.000 (±0.000), recall: 0.877 (±0.069), for {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}
precision: 0.968 (±0.039), recall: 0.780 (±0.083), for {'C': 10, 'gamma': 0.0001, 'kernel': 'rbf'}
precision: 1.000 (±0.000), recall: 0.877 (±0.069), for {'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}
precision: 0.905 (±0.058), recall: 0.889 (±0.074), for {'C': 100, 'gamma': 0.0001, 'kernel': 'rbf'}
precision: 1.000 (±0.000), recall: 0.877 (±0.069), for {'C': 1000, 'gamma': 0.001, 'kernel': 'rbf'}
precision: 0.904 (±0.058), recall: 0.890 (±0.073), for {'C': 1000, 'gamma': 0.0001, 'kernel': 'rbf'}
precision: 0.695 (±0.073), recall: 0.743 (±0.065), for {'C': 1, 'kernel': 'linear'}
precision: 0.643 (±0.066), recall: 0.757 (±0.066), for {'C': 10, 'kernel': 'linear'}
precision: 0.6

In [3]:
res = grid_search.cv_results_
print([p for p in res["params"]][:4])
train_index, test_index = train_test_split(np.arange(n_samples), test_size=0.5, random_state=0)
assert (X[train_index] == X_train).all()
nb.share(X_train=X_train, y_train=y_train.astype(np.int64), X_test=X_test)

[{'C': 1, 'gamma': 0.001, 'kernel': 'rbf'}, {'C': 1, 'gamma': 0.0001, 'kernel': 'rbf'}, {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}, {'C': 10, 'gamma': 0.0001, 'kernel': 'rbf'}]


## 2. Moxie (native Mojo)

`GridSearchCV[DigitsSearch, RefitStrategy]` with a list of two grids, two typed scorers and the custom
refit policy. Parameter keys are sorted inside each grid, as in sklearn's `ParameterGrid`.

In [4]:
%%mojo grid_search
from moxie.model_selection import (
    GridSearchCV, IndexArray, IndexSplit, SearchAdapter, SearchRefitPolicy, SearchResults,
    SearchScorer, StratifiedKFold,
)
from moxie.svm import SVC, Gamma
from moxie_nb_io import load, load_ints, save, save_floats, save_ints
from numojo import NDArray, Shape, Item

comptime Dense = NDArray[DType.float64]
comptime Labels = List[Int]
comptime Classifier = SVC[Int]


def take_rows(X: Dense, rows: IndexArray) raises -> Dense:
    var out = Dense(Shape(rows.size, X.shape[1]))
    for i in range(rows.size):
        for j in range(X.shape[1]):
            out[Item(i, j)] = X[Item(Int(rows[Item(i)]), j)]
    return out^


def take_labels(y: Labels, rows: IndexArray) raises -> Labels:
    var out = Labels()
    for i in range(rows.size):
        out.append(y[Int(rows[Item(i)])])
    return out^


@fieldwise_init
struct BinaryScore(SearchScorer):
    """sklearn's "precision" / "recall" scorers for pos_label=1 (zero_division -> 0.0)."""
    comptime Estimator = Classifier
    comptime Input = Dense
    comptime Target = Labels
    comptime Metadata = NoneType
    var recall: Bool

    def __init__(out self):
        self.recall = False

    def score(self, estimator: Classifier, X: Dense, y: Labels, metadata: NoneType) raises -> Float64:
        var pred = estimator.predict(X)
        var tp = 0
        var predicted = 0
        var actual = 0
        for i in range(len(y)):
            if pred[i] == 1:
                predicted += 1
                if y[i] == 1:
                    tp += 1
            if y[i] == 1:
                actual += 1
        var denominator = actual if self.recall else predicted
        if denominator == 0:
            return 0.0
        return Float64(tp) / Float64(denominator)


struct DigitsSearch(SearchAdapter):
    """Applies kernel (0.0 = "rbf", 1.0 = "linear"), gamma and C to a fresh SVC clone."""
    comptime Scorer = BinaryScore
    comptime Value = Float64

    def __init__(out self):
        pass

    def __init__(out self, *, copy: Self):
        pass

    def n_samples(self, X: Dense, y: Labels) raises -> Int:
        if X.ndim != 2 or X.shape[0] != len(y):
            raise Error("ValueError: mismatched rows")
        return len(y)

    def input_rows(self, X: Dense, rows: IndexArray, train: IndexArray) raises -> Dense:
        return take_rows(X, rows)

    def target_rows(self, y: Labels, rows: IndexArray) raises -> Labels:
        return take_labels(y, rows)

    def metadata_rows(self, metadata: NoneType, rows: IndexArray) raises -> NoneType:
        return None

    def fit(self, mut estimator: Classifier, X: Dense, y: Labels, metadata: NoneType) raises:
        _ = estimator.fit(X, y)

    def configure(self, mut estimator: Classifier, params: Dict[String, Float64]) raises:
        var config = estimator.get_params()
        for key in params.keys():
            if key == "C":
                config.C = params[key]
            elif key == "gamma":
                config.gamma = Gamma(params[key])
            elif key == "kernel":
                config.kernel = "rbf" if params[key] == 0.0 else "linear"
            else:
                raise Error("ValueError: Invalid parameter " + key)
        _ = estimator.set_params(config)

    def default_scorer(self) raises -> BinaryScore:
        return BinaryScore(False)

    def default_cv(self, X: Dense, y: Labels) raises -> List[IndexSplit]:
        var codes = IndexArray(Shape(len(y)))
        for i in range(len(y)):
            codes[Item(i)] = Int64(y[i])
        var cursor = StratifiedKFold(5).split(len(y), codes^)
        var out = List[IndexSplit]()
        while True:
            var pair = cursor.next()
            if not pair:
                break
            out.append(pair.value().copy())
        return out^


struct RefitStrategy(SearchRefitPolicy):
    """The example's refit_strategy: precision > 0.98, recall within one (ddof=1) std of the best,
    then the smallest mean_score_time. Metric columns: 0 = precision, 1 = recall."""
    comptime Value = Float64
    comptime Results = SearchResults[Float64]

    def __init__(out self):
        pass

    def __init__(out self, *, copy: Self):
        pass

    def survivors(self, results: Self.Results) raises -> List[Int]:
        var means = results.mean_test_score.copy()
        var high = List[Int]()
        for c in range(len(results.params)):
            if means[Item(c, 0)] > 0.98:
                high.append(c)
        var best = -1.0
        var total = 0.0
        for c in high:
            best = max(best, means[Item(c, 1)])
            total += means[Item(c, 1)]
        var mean = total / Float64(len(high))
        var squares = 0.0
        for c in high:
            squares += (means[Item(c, 1)] - mean) ** 2
        var threshold = best - (squares / Float64(len(high) - 1)) ** 0.5
        var kept = List[Int]()
        for c in high:
            if means[Item(c, 1)] > threshold:
                kept.append(c)
        return kept^

    def select(self, results: Self.Results) raises -> Int:
        var kept = self.survivors(results)
        var fastest = kept[0]
        for c in kept:
            if results.mean_score_time.item(c) < results.mean_score_time.item(fastest):
                fastest = c
        return fastest


def main() raises:
    var X_train = load("X_train")
    var y_train = load_ints("y_train")
    var X_test = load("X_test")
    var rbf = Dict[String, List[Float64]]()
    rbf["kernel"] = [0.0]
    rbf["gamma"] = [1e-3, 1e-4]
    rbf["C"] = [1.0, 10.0, 100.0, 1000.0]
    var linear = Dict[String, List[Float64]]()
    linear["kernel"] = [1.0]
    linear["C"] = [1.0, 10.0, 100.0, 1000.0]
    var grids: List[Dict[String, List[Float64]]] = [rbf^, linear^]
    var scorers: List[BinaryScore] = [BinaryScore(False), BinaryScore(True)]
    var search = GridSearchCV[DigitsSearch, RefitStrategy](
        SVC[Int](), grids, DigitsSearch(), scoring=scorers^,
        metric_names=List[String](["precision", "recall"]), refit=RefitStrategy(),
    )
    search.fit(X_train, y_train, None, None)
    var results = search.cv_results_()
    save("mean_test_score", results.mean_test_score)
    save("std_test_score", results.std_test_score)
    save("rank_test_score", results.rank_test_score)
    var keys = List[String]()
    var values = List[Float64]()
    for c in range(len(results.params)):
        for key in results.params[c].keys():
            keys.append(String(c) + ":" + key)
            values.append(results.params[c][key])
    save_floats("param_values", values)
    from moxie_nb_io import save_strings
    save_strings("param_keys", keys)
    save_ints("survivors", RefitStrategy().survivors(results))
    save_ints("best_index", [search.best_index_()])
    save_ints("y_pred", search.best_estimator_().predict(X_test))
    print("best_index_:", search.best_index_(), " n_splits_:", search.n_splits_(), " multimetric_:", search.multimetric_())

best_index_: 0  n_splits_: 5  multimetric_: True


## 3. Side by side and parity

In [5]:
from sklearn.metrics import precision_score, recall_score

mx = nb.results["grid_search"]
metrics = ["precision", "recall"]
sk_mean = np.stack([res[f"mean_test_{m}"] for m in metrics], axis=1)
sk_std = np.stack([res[f"std_test_{m}"] for m in metrics], axis=1)
sk_rank = np.stack([res[f"rank_test_{m}"] for m in metrics], axis=1)
mx_mean = np.asarray(mx["mean_test_score"]).reshape(sk_mean.shape)
moxie_nb.side_by_side(sklearn=sk_mean, moxie=mx_mean)

# Decode Moxie's candidate list back to sklearn's params dicts.
decoded = [{} for _ in res["params"]]
for key, value in zip(mx["param_keys"], mx["param_values"]):
    c, name = str(key).split(":")
    decoded[int(c)][name] = ("rbf" if value == 0.0 else "linear") if name == "kernel" else float(value)
nb.compare("params (ParameterGrid order over two grids)",
           np.array([str(sorted(p.items())) for p in res["params"]]),
           np.array([str(sorted((k, int(v) if k == "C" else v) for k, v in p.items())) for p in decoded]), exact=True)
nb.compare("mean_test_precision/recall", sk_mean, mx_mean, rtol=0, atol=1e-15,
           reason="means of identical fold scores; summation order may differ in the last bit")
nb.compare("std_test_precision/recall", sk_std, np.asarray(mx["std_test_score"]).reshape(sk_std.shape), rtol=0, atol=1e-15,
           reason="population std of identical fold scores")
nb.compare("rank_test_precision/recall", sk_rank, np.asarray(mx["rank_test_score"]).reshape(sk_rank.shape), exact=True)
nb.compare("refit_strategy survivors (deterministic filters)", survivors["index"], np.sort(np.asarray(mx["survivors"])), exact=True)
mx_best = int(mx["best_index"][0])
print("sklearn best_index_:", grid_search.best_index_, " Moxie best_index_:", mx_best,
      " (both chosen by wall-clock mean_score_time among the survivors)")
nb.compare("best_index_ is a survivor", np.array([True]), np.array([mx_best in set(survivors["index"])]), exact=True)
chosen = res["params"][mx_best]
same = SVC(**chosen).fit(X_train, y_train)
nb.compare("refitted predict(X_test) vs sklearn SVC with Moxie's chosen params",
           same.predict(X_test).astype(np.int64), np.asarray(mx["y_pred"]), exact=True)
print(classification_report(y_test, np.asarray(mx["y_pred"]).astype(bool)))
nb.note("final refit pick", "ADAPTED",
        "the official strategy ends with idxmin(mean_score_time), a wall-clock value; the pick can differ between runs on either side, so the deterministic survivor set is compared instead")
nb.note("scoring=['precision','recall']", "GAP",
        "no scorer strings or moxie.metrics (shared_surface #31); typed SearchScorer written in the cell")
nb.note("kernel string in the grid", "ADAPTED", "single grid Value type; kernel encoded as 0.0/1.0 by the typed adapter")
nb.note("classification_report", "GAP", "no moxie.metrics; computed by sklearn from Moxie's predictions")
nb.note("bool target", "ADAPTED", "SVC[Int] with 0/1 labels (typed label domain)")

sklearn best_index_: 6  Moxie best_index_: 0  (both chosen by wall-clock mean_score_time among the survivors)
              precision    recall  f1-score   support

       False       0.98      1.00      0.99       807
        True       1.00      0.85      0.92        92

    accuracy                           0.98       899
   macro avg       0.99      0.92      0.95       899
weighted avg       0.98      0.98      0.98       899



In [6]:
nb.report()
nb.assert_parity()

check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
params (ParameterGrid order over two grids),PASS,"(12,)","(12,)",,,exact,0 mismatches,
mean_test_precision/recall,PASS,"(12, 2)","(12, 2)",0,0,"rtol=0, atol=1e-15",,means of identical fold scores; summation order may differ in the last bit
std_test_precision/recall,PASS,"(12, 2)","(12, 2)",0,0,"rtol=0, atol=1e-15",,population std of identical fold scores
rank_test_precision/recall,PASS,"(12, 2)","(12, 2)",,,exact,0 mismatches,
refit_strategy survivors (deterministic filters),PASS,"(4,)","(4,)",,,exact,0 mismatches,
best_index_ is a survivor,PASS,"(1,)","(1,)",,,exact,0 mismatches,
refitted predict(X_test) vs sklearn SVC with Moxie's chosen params,PASS,"(899,)","(899,)",,,exact,0 mismatches,
final refit pick,ADAPTED,,,,,,"the official strategy ends with idxmin(mean_score_time), a wall-clock value; the pick can differ between runs on either side, so the deterministic survivor set is compared instead",
"scoring=['precision','recall']",GAP,,,,,,no scorer strings or moxie.metrics (shared_surface #31); typed SearchScorer written in the cell,
kernel string in the grid,ADAPTED,,,,,,single grid Value type; kernel encoded as 0.0/1.0 by the typed adapter,
